# Roadmap

1. Preprocessing con VAD para eliminar ruidos o momentos de silencio.
2. Ventanas con solapamiento.
3. Extracción de características (OpenSMILE, Librosa), según fuentes: Prosody, voice quality, spectral features (MFCC). O modelos como Wav2Vec.
4. Modelización y evaluación.

5. Usar LLMs multimodales para analizar la entrada de audio directamente.

In [ ]:
import torch
import os

# Load and preprocessing

In [ ]:
#VAD model
model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad', model='silero_vad', force_reload=True)
(get_speech_timestamps,save_audio,read_audio,VADIterator,collect_chunks) = utils
sampling_rate = 16000

Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /home/ethe/.cache/torch/hub/master.zip


In [19]:
audio = './data/original/2ea4/2ea4_Counting3.wav'

wav = read_audio(audio, sampling_rate=sampling_rate)
# get speech timestamps from full audio file
speech_timestamps = get_speech_timestamps(wav, 
                                        model, 
                                        sampling_rate=sampling_rate,
                                        threshold = 0.3,
                                        speech_pad_ms = 250,
                                        )


save_audio('example.wav', collect_chunks(speech_timestamps, wav), sampling_rate=sampling_rate)

/home/ethe/miniconda3/envs/tfg/lib/python3.11/site-packages/torchaudio/__init__.py:178: UserWarning: The 'bits_per_sample' parameter is not directly supported by TorchCodec AudioEncoder.
  return save_with_torchcodec(


In [20]:
from IPython.display import Audio

Audio('example.wav')

In [21]:
Audio(audio)

In [30]:
#VAD model
#model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad', model='silero_vad', force_reload=True)
#(get_speech_timestamps,save_audio,read_audio,VADIterator,collect_chunks) = utils

def apply_vad(input_dir, output_dir, sampling_rate = 16000, threshold = 0.2, speech_pad_ms=250):

    for subject in os.listdir(input_dir):

        for act in os.listdir(os.path.join(input_dir, subject)):

            audio_path = os.path.join(input_dir, subject, act)
            subject_act = os.path.basename(audio_path).split('.')[0]
            wav = read_audio(audio_path, sampling_rate=sampling_rate)

            speech_timestamps = get_speech_timestamps(wav, 
                                        model, 
                                        sampling_rate=sampling_rate,
                                        threshold = threshold,
                                        speech_pad_ms = speech_pad_ms,
                                        )
            
            if len(speech_timestamps) > 0:
                out_dir = os.path.join(output_dir, subject)
                os.makedirs(out_dir, exist_ok=True)
                save_audio(os.path.join(out_dir, f"{subject_act}.wav"), collect_chunks(speech_timestamps, wav), sampling_rate=sampling_rate)

            else:
                print('VAD failed on', audio_path)

apply_vad(input_dir='./data/original/', output_dir='./data/vad_applied')

/home/ethe/miniconda3/envs/tfg/lib/python3.11/site-packages/torchaudio/__init__.py:178: UserWarning: The 'bits_per_sample' parameter is not directly supported by TorchCodec AudioEncoder.
  return save_with_torchcodec(


VAD failed on ./data/original/h7j3/h7j3_Stroop.wav
VAD failed on ./data/original/h7j3/h7j3_Counting3.wav
VAD failed on ./data/original/h7j3/h7j3_Counting1.wav
VAD failed on ./data/original/h7j3/h7j3_Math.wav


# Windowing

In [2]:
import os
import numpy as np
import soundfile as sf
import librosa


def save_audio_windows(input_dir, output_dir, window_size=5.0, overlap=0.0, sampling_rate=16000):
    """
    Splits each audio file into fixed-length windows and saves them.

    input_dir structure:
        input_dir/subject/activity.wav

    output_dir structure:
        output_dir/subject/activity/window_0.wav
    """

    stride = window_size * (1 - overlap)

    for subject in sorted(os.listdir(input_dir)):
        subject_dir = os.path.join(input_dir, subject)

        if not os.path.isdir(subject_dir):
            continue

        for act_file in sorted(os.listdir(subject_dir)):
            if not act_file.lower().endswith(".wav"):
                continue

            audio_path = os.path.join(subject_dir, act_file)
            activity = os.path.splitext(act_file)[0]

            y, sr = librosa.load(audio_path, sr=sampling_rate)
            duration = librosa.get_duration(y=y, sr=sr)

            out_act_dir = os.path.join(output_dir, subject, activity)
            os.makedirs(out_act_dir, exist_ok=True)

            start = 0.0
            window_id = 0

            while start + window_size <= duration:
                end = start + window_size

                start_sample = int(start * sr)
                end_sample = int(end * sr)

                y_window = y[start_sample:end_sample]

                out_path = os.path.join(out_act_dir, f"window_{window_id}.wav")
                sf.write(out_path, y_window, sr)

                window_id += 1
                start += stride

            if window_id == 0:
                print(f"No complete windows for: {audio_path}")

        print(f"{subject} done.")

    print("All audio windows saved.")

In [3]:
save_audio_windows(input_dir='./data/vad_applied', output_dir='./data/5s_0overlap')

/home/ethe/miniconda3/envs/tfg/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2ea4 done.
2hpu done.
2z7d done.
45lx done.
4e8r done.
4woj done.
5f7t done.
6g6y done.
6k5f done.
71i5 done.
7h5u done.
7m3c done.
No complete windows for: ./data/vad_applied/8g4y/8g4y_Math.wav
No complete windows for: ./data/vad_applied/8g4y/8g4y_Stroop.wav
8g4y done.
No complete windows for: ./data/vad_applied/8i4i/8i4i_Counting3.wav
No complete windows for: ./data/vad_applied/8i4i/8i4i_Math.wav
No complete windows for: ./data/vad_applied/8i4i/8i4i_Stroop.wav
8i4i done.
9j3o done.
No complete windows for: ./data/vad_applied/9t6n/9t6n_Counting2.wav
9t6n done.
9txq done.
a1k9 done.
b2l8 done.
b9w0 done.
bfl5 done.
c3m7 done.
chdf done.
ctzy done.
cxj0 done.
No complete windows for: ./data/vad_applied/d4n6/d4n6_Stroop.wav
d4n6 done.
e5p4 done.
g7r2 done.
g9j5 done.
No complete windows for: ./data/vad_applied/h7j3/h7j3_Counting2.wav
h7j3 done.
h8r2 done.
h8s1 done.
i9t9 done.
iqyg done.
No complete windows for: ./data/vad_applied/j9h8/j9h8_Counting3.wav
j9h8 done.
k2v7 done.
k67g done.


# Paralinguistic audio features extraction using LIBROSA:

In [ ]:
import pandas as pd

def extract_features_from_audio(y, sr):
    features = {}

    # Energy / loudness
    rms = librosa.feature.rms(y=y)[0]
    features["rms_mean"] = np.mean(rms)
    features["rms_std"] = np.std(rms)

    # Zero-crossing rate
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features["zcr_mean"] = np.mean(zcr)
    features["zcr_std"] = np.std(zcr)

    # MFCCs
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

    for i in range(13):
        features[f"mfcc_{i+1}_mean"] = np.mean(mfcc[i])
        features[f"mfcc_{i+1}_std"] = np.std(mfcc[i])

    # Pitch / F0
    f0, voiced_flag, voiced_probs = librosa.pyin(
        y,
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C7")
    )

    valid_f0 = f0[~np.isnan(f0)]

    if len(valid_f0) > 0:
        features["pitch_mean"] = np.mean(valid_f0)
        features["pitch_std"] = np.std(valid_f0)
        features["pitch_min"] = np.min(valid_f0)
        features["pitch_max"] = np.max(valid_f0)
    else:
        features["pitch_mean"] = 0.0
        features["pitch_std"] = 0.0
        features["pitch_min"] = 0.0
        features["pitch_max"] = 0.0

    # Voiced ratio
    features["voiced_ratio"] = np.mean(voiced_flag) if voiced_flag is not None else 0.0

    # Pause / low-energy ratio
    if len(rms) > 0:
        silence_threshold = np.percentile(rms, 25)
        features["low_energy_ratio"] = np.mean(rms < silence_threshold)
    else:
        features["low_energy_ratio"] = 0.0

    return features

def extract_audio_window_features(windows_dir, output_csv, sampling_rate=16000):
    """
    Iterates through saved audio windows and extracts acoustic features.

    windows_dir structure:
        windows_dir/subject/activity/window_0.wav
    """

    rows = []

    for subject in sorted(os.listdir(windows_dir)):
        subject_dir = os.path.join(windows_dir, subject)

        if not os.path.isdir(subject_dir):
            continue

        for activity in sorted(os.listdir(subject_dir)):
            activity_dir = os.path.join(subject_dir, activity)

            if not os.path.isdir(activity_dir):
                continue

            for window_file in sorted(os.listdir(activity_dir)):
                if not window_file.lower().endswith(".wav"):
                    continue

                window_path = os.path.join(activity_dir, window_file)

                window_id = int(
                    os.path.splitext(window_file)[0].replace("window_", "")
                )

                y, sr = librosa.load(window_path, sr=sampling_rate)

                features = extract_features_from_audio(y, sr)

                rows.append({
                    "subject": subject,
                    "activity": activity.split('_')[1],
                    "window_id": window_id,
                    "subject_activity": activity,
                    **features
                })

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)

    print(f"Saved features to: {output_csv}")
    print(f"Total windows processed: {len(df)}")

    return df

In [6]:
audio_features = extract_audio_window_features(
    windows_dir="./data/5s_0overlap",
    output_csv="audio_features_5s_0overlap.csv",
    sampling_rate=16000
)

Saved features to: audio_features_5s_0overlap.csv
Total windows processed: 3091


In [31]:
df_features = pd.read_csv('audio_features_5s_0overlap.csv')
df_features.isna().sum()

subject             0
activity            0
window_id           0
subject_activity    0
rms_mean            0
rms_std             0
zcr_mean            0
zcr_std             0
mfcc_1_mean         0
mfcc_1_std          0
mfcc_2_mean         0
mfcc_2_std          0
mfcc_3_mean         0
mfcc_3_std          0
mfcc_4_mean         0
mfcc_4_std          0
mfcc_5_mean         0
mfcc_5_std          0
mfcc_6_mean         0
mfcc_6_std          0
mfcc_7_mean         0
mfcc_7_std          0
mfcc_8_mean         0
mfcc_8_std          0
mfcc_9_mean         0
mfcc_9_std          0
mfcc_10_mean        0
mfcc_10_std         0
mfcc_11_mean        0
mfcc_11_std         0
mfcc_12_mean        0
mfcc_12_std         0
mfcc_13_mean        0
mfcc_13_std         0
pitch_mean          0
pitch_std           0
pitch_min           0
pitch_max           0
voiced_ratio        0
low_energy_ratio    0
dtype: int64

# Per task aggregation:

In [49]:
def task_aggregation(df_audio, gt):
    df = df_audio.copy()

    # Remove non-feature columns
    id_cols = ["subject", "activity", "window_id", "subject_activity"]
    feature_cols = [c for c in df.columns if c not in id_cols]

    # Make sure features are numeric
    df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")

    # Aggregate window-level features to task-level
    agg = df.groupby("subject_activity")[feature_cols].agg(["mean", "std", "max"])
    agg.columns = [f"{feat}_{stat}" for feat, stat in agg.columns]
    agg = agg.reset_index()

    # Merge with GT
    dataset = gt.merge(agg, on="subject_activity", how="inner")

    X = dataset.drop(columns=["subject_activity", "y_true"])
    y = dataset["y_true"].astype(int)

    return X, y, dataset

gt = pd.read_csv('../labels_v2.csv', sep=';').rename(columns={'subject/task':'subject_activity', 'binary-stress':'y_true'}).drop(['affect3-class', 'affect3-class-v2'], axis=1)
gt

,subject_activity,y_true
0,2ea4_Breathing,0
1,2ea4_Counting1,1
2,2ea4_Counting2,1
3,2ea4_Counting3,1
4,2ea4_Math,1
...,...,...
695,y9z6_Relax,0
696,y9z6_Speaking,1
697,y9z6_Stroop,1
698,y9z6_Video1,1


In [50]:
X, y, agg_df = task_aggregation(df_features, gt)

In [57]:
X.isna().sum()

rms_mean_mean             0
rms_mean_std             10
rms_mean_max              0
rms_std_mean              0
rms_std_std              10
                         ..
voiced_ratio_std         10
voiced_ratio_max          0
low_energy_ratio_mean     0
low_energy_ratio_std     10
low_energy_ratio_max      0
Length: 108, dtype: int64

In [59]:
nan_cols = agg_df.isna().sum()
nan_cols[nan_cols > 0].sort_values(ascending=False)

rms_mean_std            10
rms_std_std             10
zcr_mean_std            10
zcr_std_std             10
mfcc_1_mean_std         10
mfcc_1_std_std          10
mfcc_2_mean_std         10
mfcc_2_std_std          10
mfcc_3_mean_std         10
mfcc_3_std_std          10
mfcc_4_mean_std         10
mfcc_4_std_std          10
mfcc_5_mean_std         10
mfcc_5_std_std          10
mfcc_6_mean_std         10
mfcc_6_std_std          10
mfcc_7_mean_std         10
mfcc_7_std_std          10
mfcc_8_mean_std         10
mfcc_8_std_std          10
mfcc_9_mean_std         10
mfcc_9_std_std          10
mfcc_10_mean_std        10
mfcc_10_std_std         10
mfcc_11_mean_std        10
mfcc_11_std_std         10
mfcc_12_mean_std        10
mfcc_12_std_std         10
mfcc_13_mean_std        10
mfcc_13_std_std         10
pitch_mean_std          10
pitch_std_std           10
pitch_min_std           10
pitch_max_std           10
voiced_ratio_std        10
low_energy_ratio_std    10
dtype: int64

Estos son los casos en que solo se pudo extraer 1 ventana, por lo que no se pudo computar la desviacion. Podemos imputarlos con 0, refiriendo a que no hay cambios entre ventanas.

In [62]:
std_cols = [c for c in agg_df.columns if c.endswith("_std")]
agg_df[std_cols] = agg_df[std_cols].fillna(0)
X[std_cols] = X[std_cols].fillna(0)
nan_cols = agg_df.isna().sum()
nan_cols[nan_cols > 0].sort_values(ascending=False)

Series([], dtype: int64)

In [63]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

clf = Pipeline([
    #("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_validate(
    clf,
    X,
    y,
    cv=cv,
    scoring=["accuracy", "balanced_accuracy", "f1", "precision", "recall"],
)

pd.DataFrame(scores).mean()

fit_time                  0.008908
score_time                0.005690
test_accuracy             0.517960
test_balanced_accuracy    0.476774
test_f1                   0.629096
test_precision            0.701607
test_recall               0.571644
dtype: float64

In [64]:
y.value_counts(normalize=True)

y_true
1    0.717452
0    0.282548
Name: proportion, dtype: float64